# Experiment 06 — Repeated-run stability

Current gate under `ROADMAP_REVISED_V2`. Repeat only non-private, ε≈4, and ε≈2
with seeds `[42, 52, 62, 72, 82]`. Reuse accepted seed 42; normally train 12 new
target models. No new shadows are trained.

**Run:** open this version in Colab, select a CPU runtime (hardware accelerator: None),
then **Runtime → Run all** and authorize access to the existing `ML-DP-NID` Drive folder.
Dependencies are pinned to Experiment 05. If a package was already imported at a different
version, restart the session once and Run all again, as instructed by the environment check.

Before new training, the notebook verifies accepted artifacts, reconstructs frozen attackers,
and reproduces seed-42 MIA results. Missing accepted inputs stop the run with recovery instructions;
they do not trigger a rerun of Experiment 05. Valid Experiment 06 caches resume automatically.

**Primary attacks:** score-only logistic regression and label-aware loss threshold, fixed
across conditions. **Secondary:** the best Experiment 05 shadow-calibrated attacker for each
condition/threat model. Both policies are frozen before any new target is trained.

The final gate downloads `experiment06_evidence.zip`. Send that ZIP for scientific review.
Passing the technical gate does not establish ε≈4 as optimal or demonstrate leakage reduction.
This experiment measures target-seed variability conditional on fixed splits and attackers.


## 1. Install and import dependencies


In [19]:
# Match the accepted Experiment 05 packages before importing numerical libraries.
import importlib.metadata
import subprocess
import sys

PINNED_PACKAGES = {
    "numpy": "2.1.3", "pandas": "2.2.3", "scikit-learn": "1.6.1",
    "torch": "2.11.0", "opacus": "1.6.0",
}
def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

if installed_version("torch") != "2.11.0+cpu":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "torch==2.11.0+cpu",
        "--index-url", "https://download.pytorch.org/whl/cpu",
    ])
missing_or_different = [
    f"{name}=={version}" for name, version in PINNED_PACKAGES.items()
    if name != "torch" and installed_version(name) != version
]
if missing_or_different:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing_or_different,
    ])
print("Experiment 05 package versions installed; checking imported runtime next.")


Experiment 05 package versions installed; checking imported runtime next.


In [20]:
from __future__ import annotations

import copy
import gc
import hashlib
import json
import math
import os
import platform
import random
import time
import warnings
import zipfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

pd.set_option("display.max_columns", 120)

print({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "opacus": __import__("opacus").__version__,
    "sklearn": sklearn.__version__,
})

RUNTIME_IDENTITY = {
    "python": sys.version.split()[0], "numpy": np.__version__,
    "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
    "torch": torch.__version__, "opacus": __import__("opacus").__version__,
    "device": "cpu",
}
runtime_mismatches = {
    name: {"expected": version, "loaded": RUNTIME_IDENTITY[name]}
    for name, version in PINNED_PACKAGES.items()
    if RUNTIME_IDENTITY[name].split("+")[0] != version
}
if RUNTIME_IDENTITY["torch"] != "2.11.0+cpu":
    runtime_mismatches["torch_build"] = {"expected": "2.11.0+cpu", "loaded": RUNTIME_IDENTITY["torch"]}
assert not runtime_mismatches, (
    "Packages were updated after this session had imported older versions. "
    "Restart the Colab session, then Run all again. " + str(runtime_mismatches)
)


{'python': '3.13.15', 'torch': '2.11.0+cpu', 'opacus': '1.6.0', 'sklearn': '1.6.1'}


## 2. Locked Experiment 06 configuration


In [21]:
PROTOCOL_VERSION = "ROADMAP_REVISED_V2"
PROTOCOL_REVISION = "repeated_target_seeds_fixed_attackers_v2"

BASE_SEED = 42
RUN_SEEDS = [42, 52, 62, 72, 82]
EXPERIMENT_05_SEED = 42
SHADOW_SEEDS = [101, 202, 303, 404, 505]
CALIBRATION_SHADOW_SEED = 505

HIDDEN_DIMS = (64, 32)
EPOCHS = 30
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

MAX_GRAD_NORM = 1.0
ACCOUNTANT = "prv"
POISSON_SAMPLING = True
SECURE_MODE = False

RESUME = True
REUSE_ACCEPTED_SEED_42 = True
T_CRITICAL_95_DF4 = 2.7764451051977987

CONDITIONS = [
    {"condition": "non_private", "formal_dp": False, "target_epsilon": None},
    {"condition": "dp_eps_4", "formal_dp": True, "target_epsilon": 4.0},
    {"condition": "dp_eps_2", "formal_dp": True, "target_epsilon": 2.0},
]

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

def stable_seed(*parts: object) -> int:
    value = "::".join(str(part) for part in parts)
    return int(hashlib.sha256(value.encode("utf-8")).hexdigest()[:8], 16)

set_all_seeds(BASE_SEED)
DEVICE = torch.device("cpu")

display(pd.DataFrame(CONDITIONS))
print({"seeds": RUN_SEEDS, "device": str(DEVICE)})

PRIMARY_ATTACK_MODELS = {
    "score_only_black_box": "logistic_regression",
    "label_aware_audit": "loss_threshold",
}
IDS_SUMMARY_METRICS = ["recall", "fnr", "f1", "fpr", "precision", "pr_auc", "threshold"]
MIA_SUMMARY_METRICS = [
    "mia_auc", "mia_advantage", "mia_balanced_accuracy",
    "tpr_at_1pct_fpr", "tpr_at_5pct_fpr",
]


,condition,formal_dp,target_epsilon
0,non_private,False,NaN
1,dp_eps_4,True,4.0
2,dp_eps_2,True,2.0


{'seeds': [42, 52, 62, 72, 82], 'device': 'cpu'}


## 3. Resolve Drive paths and accepted Experiment 05 prerequisites


In [22]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

DEFAULT_PROJECT_DIR = Path("/content/drive/MyDrive/ML-DP-NID")
PROJECT_DIR = Path(os.environ.get("ML_DP_NID_DIR", DEFAULT_PROJECT_DIR))
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()

TRAIN_FILE = PROJECT_DIR / "KDDTrain+.txt"
TEST_FILE = PROJECT_DIR / "KDDTest+.txt"

SPLIT_DIR = PROJECT_DIR / "data" / "split_indices"
TARGET_TRAIN_INDEX_FILE = SPLIT_DIR / "target_train_indices.npy"
TARGET_VALIDATION_INDEX_FILE = SPLIT_DIR / "target_validation_indices.npy"
SHADOW_POOL_INDEX_FILE = SPLIT_DIR / "shadow_pool_indices.npy"
PREPROCESSOR_FILE = PROJECT_DIR / "artifacts" / "preprocessor" / "target_mlp_preprocessor.joblib"

EXP05_RESULTS_DIR = PROJECT_DIR / "results" / "dp_sgd"
EXP05_INTERMEDIATE_DIR = EXP05_RESULTS_DIR / "intermediate"
EXP05_MANIFEST_FILE = EXP05_RESULTS_DIR / "dp_sgd_sweep_manifest.json"
EXP05_CONFIG_FILE = EXP05_RESULTS_DIR / "config.json"
EXP05_IDS_FILE = EXP05_RESULTS_DIR / "dp_sgd_ids_results.csv"
EXP05_MIA_FILE = EXP05_RESULTS_DIR / "dp_sgd_mia_results.csv"
EXP05_TARGET_CONFIGS_FILE = EXP05_RESULTS_DIR / "dp_sgd_configs.csv"
EXP05_ATTACK_CALIBRATION_FILE = EXP05_RESULTS_DIR / "mia_attack_calibration.csv"
EXP05_TARGET_SAMPLE_FILE = EXP05_RESULTS_DIR / "target_mia_sample_manifest.csv"
EXP05_WARNING_FILE = EXP05_RESULTS_DIR / "opacus_warning_summary.csv"

RESULTS_DIR = PROJECT_DIR / "results" / "repeated_runs"
INTERMEDIATE_DIR = RESULTS_DIR / "intermediate"
MODEL_DIR = PROJECT_DIR / "artifacts" / "models" / "repeated_runs"
MANIFEST_DIR = PROJECT_DIR / "artifacts" / "manifests"

for directory in [RESULTS_DIR, INTERMEDIATE_DIR, MODEL_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

required_files = [
    TRAIN_FILE,
    TEST_FILE,
    TARGET_TRAIN_INDEX_FILE,
    TARGET_VALIDATION_INDEX_FILE,
    SHADOW_POOL_INDEX_FILE,
    PREPROCESSOR_FILE,
    EXP05_MANIFEST_FILE,
    EXP05_CONFIG_FILE,
    EXP05_IDS_FILE,
    EXP05_MIA_FILE,
    EXP05_TARGET_CONFIGS_FILE,
    EXP05_ATTACK_CALIBRATION_FILE,
    EXP05_TARGET_SAMPLE_FILE,
    EXP05_WARNING_FILE,
]
required_files.extend(
    EXP05_INTERMEDIATE_DIR / spec["condition"] / "shadow_attack_data.csv"
    for spec in CONDITIONS
)

missing_files = [str(path) for path in required_files if not path.exists()]
assert not missing_files, (
    "Missing accepted Experiment 05 prerequisites. Recover these files from the original ML-DP-NID run; do not rerun the sweep:\n"
    + "\n".join(missing_files)
)

print("Project directory:", PROJECT_DIR)
print("Experiment 06 result directory:", RESULTS_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/ML-DP-NID
Experiment 06 result directory: /content/drive/MyDrive/ML-DP-NID/results/repeated_runs


In [23]:
def calculate_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def load_json(path: Path) -> dict:
    with path.open("r") as file:
        return json.load(file)

def to_json_safe(value):
    if isinstance(value, dict):
        return {str(key): to_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_safe(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

def write_strict_json(path: Path, payload: dict) -> None:
    with path.open("w") as file:
        json.dump(to_json_safe(payload), file, indent=2, allow_nan=False)

def canonical_hash(payload: dict) -> str:
    encoded = json.dumps(
        to_json_safe(payload),
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

exp05_manifest = load_json(EXP05_MANIFEST_FILE)
exp05_config = load_json(EXP05_CONFIG_FILE)

actual_train_hash = calculate_sha256(TRAIN_FILE)
actual_test_hash = calculate_sha256(TEST_FILE)
preprocessor_hash = calculate_sha256(PREPROCESSOR_FILE)
split_hashes = {
    "target_train_indices_sha256": calculate_sha256(TARGET_TRAIN_INDEX_FILE),
    "target_validation_indices_sha256": calculate_sha256(TARGET_VALIDATION_INDEX_FILE),
    "shadow_pool_indices_sha256": calculate_sha256(SHADOW_POOL_INDEX_FILE),
}

assert exp05_manifest["protocol_version"] == PROTOCOL_VERSION
assert exp05_manifest["protocol_revision"] == "epsilon_delta_matched_v2"
assert exp05_manifest["experiment"] == "05_dp_sgd_privacy_utility_sweep"
assert exp05_manifest["seed"] == EXPERIMENT_05_SEED
assert exp05_manifest["dataset"]["train_sha256"] == actual_train_hash
assert exp05_manifest["dataset"]["test_sha256"] == actual_test_hash
assert exp05_manifest["preprocessor"]["sha256"] == preprocessor_hash
assert exp05_manifest["architecture"]["hidden_dims"] == list(HIDDEN_DIMS)
assert exp05_manifest["training"]["epochs"] == EPOCHS
assert exp05_manifest["training"]["batch_size"] == BATCH_SIZE
assert np.isclose(exp05_manifest["training"]["learning_rate"], LEARNING_RATE)
assert np.isclose(exp05_manifest["training"]["weight_decay"], WEIGHT_DECAY)
assert exp05_manifest["dp"]["accountant"] == ACCOUNTANT
assert np.isclose(exp05_manifest["dp"]["max_grad_norm"], MAX_GRAD_NORM)
assert exp05_manifest["dp"]["poisson_sampling"] is POISSON_SAMPLING
assert exp05_manifest["dp"]["secure_mode"] is SECURE_MODE
assert exp05_config["experiment"] == exp05_manifest["experiment"]

print("Accepted Experiment 05 manifest and locked artifacts verified.")

if RUNTIME_IDENTITY["python"] != exp05_manifest["software"]["python"].split()[0]:
    print("Python version differs from Experiment 05; numerical package versions are locked. "
          "The manifest records this runtime difference and seed-42 reconstruction must pass.")


Accepted Experiment 05 manifest and locked artifacts verified.


## 4. Load NSL-KDD, the locked split, preprocessing, and target MIA records


In [24]:
COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "label", "difficulty",
]

FEATURES = COLUMNS[:41]
CATEGORICAL_FEATURES = ["protocol_type", "service", "flag"]
NUMERIC_FEATURES = [column for column in FEATURES if column not in CATEGORICAL_FEATURES]

DOS_ATTACKS = {
    "back", "land", "neptune", "pod", "smurf", "teardrop",
    "apache2", "udpstorm", "processtable", "mailbomb",
}
PROBE_ATTACKS = {"satan", "ipsweep", "nmap", "portsweep", "mscan", "saint"}
R2L_ATTACKS = {
    "guess_passwd", "ftp_write", "imap", "phf", "multihop",
    "warezmaster", "warezclient", "spy", "xlock", "xsnoop",
    "snmpguess", "snmpgetattack", "httptunnel", "sendmail", "named",
}
U2R_ATTACKS = {
    "buffer_overflow", "loadmodule", "rootkit", "perl", "sqlattack", "xterm", "ps"
}

def map_attack_family(label: str) -> str:
    label = str(label).strip().lower().rstrip(".")
    if label == "normal":
        return "Normal"
    if label in DOS_ATTACKS:
        return "DoS"
    if label in PROBE_ATTACKS:
        return "Probe"
    if label in R2L_ATTACKS:
        return "R2L"
    if label in U2R_ATTACKS:
        return "U2R"
    return "OtherAttack"

def pool_rare_families(attack_family: str) -> str:
    return attack_family if attack_family in {"Normal", "DoS", "Probe"} else "Rare"

def load_nsl_kdd(path: Path) -> pd.DataFrame:
    dataframe = pd.read_csv(path, names=COLUMNS)
    dataframe["row_id"] = np.arange(len(dataframe))
    dataframe["label_clean"] = (
        dataframe["label"].astype(str).str.strip().str.lower().str.rstrip(".")
    )
    dataframe["binary_label"] = (dataframe["label_clean"] != "normal").astype(int)
    dataframe["attack_family"] = dataframe["label_clean"].map(map_attack_family)
    dataframe["family_group"] = dataframe["attack_family"].map(pool_rare_families)
    for column in NUMERIC_FEATURES:
        dataframe[column] = pd.to_numeric(dataframe[column], errors="raise")
    return dataframe

def to_float32_dense(values) -> np.ndarray:
    if hasattr(values, "toarray"):
        values = values.toarray()
    return np.asarray(values, dtype=np.float32)


In [25]:
train_df = load_nsl_kdd(TRAIN_FILE)
test_df = load_nsl_kdd(TEST_FILE)

target_train_indices = np.load(TARGET_TRAIN_INDEX_FILE)
target_validation_indices = np.load(TARGET_VALIDATION_INDEX_FILE)
shadow_pool_indices = np.load(SHADOW_POOL_INDEX_FILE)

assert len(target_train_indices) == 88181
assert len(target_validation_indices) == 12597
assert len(shadow_pool_indices) == 25195

all_indices = np.concatenate([
    target_train_indices,
    target_validation_indices,
    shadow_pool_indices,
])
assert len(np.unique(all_indices)) == len(train_df)
assert set(all_indices) == set(range(len(train_df)))

target_train = train_df.iloc[target_train_indices].copy().reset_index(drop=True)
target_validation = train_df.iloc[target_validation_indices].copy().reset_index(drop=True)

target_preprocessor = joblib.load(PREPROCESSOR_FILE)
X_target_train = to_float32_dense(target_preprocessor.transform(target_train[FEATURES]))
X_target_validation = to_float32_dense(
    target_preprocessor.transform(target_validation[FEATURES])
)
X_test = to_float32_dense(target_preprocessor.transform(test_df[FEATURES]))

y_target_train = target_train["binary_label"].to_numpy(dtype=np.float32).reshape(-1, 1)
y_target_validation = (
    target_validation["binary_label"].to_numpy(dtype=np.float32).reshape(-1, 1)
)
y_test = test_df["binary_label"].to_numpy(dtype=np.float32).reshape(-1, 1)

TARGET_INPUT_DIM = X_target_train.shape[1]
TARGET_DELTA = 1.0 / len(X_target_train)

assert TARGET_INPUT_DIM == exp05_manifest["architecture"]["target_input_dim"]
assert np.isclose(TARGET_DELTA, exp05_manifest["target_delta"], rtol=0, atol=1e-15)

target_mia_sample_manifest = pd.read_csv(EXP05_TARGET_SAMPLE_FILE)
expected_sample_columns = {
    "membership", "partition", "partition_position", "row_id",
    "true_label", "binary_group", "family_group",
}
assert expected_sample_columns.issubset(target_mia_sample_manifest.columns)
assert len(target_mia_sample_manifest) == 25194
assert target_mia_sample_manifest["membership"].value_counts().to_dict() == {
    1: 12597,
    0: 12597,
}

for membership, partition_name, partition_frame in [
    (1, "target_train", target_train),
    (0, "target_validation", target_validation),
]:
    sample_rows = target_mia_sample_manifest[
        target_mia_sample_manifest["membership"] == membership
    ]
    assert set(sample_rows["partition"]) == {partition_name}
    positions = sample_rows["partition_position"].to_numpy(dtype=int)
    selected = partition_frame.iloc[positions]
    assert np.array_equal(
        selected["row_id"].to_numpy(dtype=int),
        sample_rows["row_id"].to_numpy(dtype=int),
    )
    assert np.array_equal(
        selected["binary_label"].to_numpy(dtype=int),
        sample_rows["true_label"].to_numpy(dtype=int),
    )
    assert np.array_equal(
        selected["family_group"].to_numpy(),
        sample_rows["family_group"].to_numpy(),
    )

target_member_positions = target_mia_sample_manifest.loc[
    target_mia_sample_manifest["membership"] == 1,
    "partition_position",
].to_numpy(dtype=int)
target_nonmember_positions = target_mia_sample_manifest.loc[
    target_mia_sample_manifest["membership"] == 0,
    "partition_position",
].to_numpy(dtype=int)

target_sample_output = RESULTS_DIR / "target_mia_sample_manifest.csv"
target_mia_sample_manifest.to_csv(target_sample_output, index=False)
target_sample_hash = calculate_sha256(target_sample_output)

print({
    "target_input_dim": TARGET_INPUT_DIM,
    "target_delta": TARGET_DELTA,
    "target_mia_rows": len(target_mia_sample_manifest),
})


{'target_input_dim': 122, 'target_delta': 1.134031140495118e-05, 'target_mia_rows': 25194}


## 5. Target MLP, training, IDS, and target-MIA helpers


In [26]:
class BinaryMLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: tuple[int, int] = HIDDEN_DIMS,
    ) -> None:
        super().__init__()
        hidden_1, hidden_2 = hidden_dims
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_1),
            nn.ReLU(),
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),
            nn.Linear(hidden_2, 1),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)

def create_initial_state(input_dim: int, seed: int) -> dict:
    set_all_seeds(seed)
    return copy.deepcopy(BinaryMLP(input_dim).state_dict())

def unwrap_model(model: nn.Module) -> nn.Module:
    return getattr(model, "_module", model)

def create_dataset(features: np.ndarray, labels: np.ndarray) -> TensorDataset:
    return TensorDataset(torch.from_numpy(features), torch.from_numpy(labels))

def create_train_loader(
    features: np.ndarray,
    labels: np.ndarray,
    seed: int,
) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        create_dataset(features, labels),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
        generator=generator,
        drop_last=False,
    )

def create_eval_loader(features: np.ndarray, labels: np.ndarray) -> DataLoader:
    return DataLoader(
        create_dataset(features, labels),
        batch_size=1024,
        shuffle=False,
        num_workers=0,
    )

def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    model.train()
    total_weighted_loss = 0.0
    total_examples = 0

    for features, labels in data_loader:
        features = features.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.shape[0]
        total_weighted_loss += float(loss.detach().cpu()) * batch_size
        total_examples += batch_size

    return total_weighted_loss / max(total_examples, 1)

@torch.no_grad()
def predict_probabilities(
    model: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
    device: torch.device,
) -> np.ndarray:
    model.eval()
    batches = []
    for batch_features, _ in create_eval_loader(features, labels):
        batch_features = batch_features.to(device, non_blocking=True)
        batches.append(torch.sigmoid(model(batch_features)).cpu().numpy())
    return np.concatenate(batches, axis=0).reshape(-1)

def summarize_warnings(
    caught_warnings,
    stage: str,
    condition: str,
    seed: int,
) -> list[dict]:
    warning_counter = Counter(
        (warning.category.__name__, str(warning.message))
        for warning in caught_warnings
    )
    return [
        {
            "stage": stage,
            "condition": condition,
            "seed": seed,
            "category": category,
            "message": message,
            "occurrences": count,
        }
        for (category, message), count in warning_counter.items()
    ]

compatibility_errors = ModuleValidator.validate(
    BinaryMLP(TARGET_INPUT_DIM),
    strict=False,
)
assert not compatibility_errors
print("Target MLP is Opacus-compatible.")


Target MLP is Opacus-compatible.


In [27]:
def select_f2_threshold(
    y_true: np.ndarray,
    probabilities: np.ndarray,
) -> tuple[float, pd.DataFrame]:
    rows = []
    for threshold in np.arange(0.01, 1.00, 0.01):
        predictions = (probabilities >= threshold).astype(int)
        rows.append({
            "threshold": float(threshold),
            "f2": fbeta_score(y_true, predictions, beta=2, zero_division=0),
            "recall": recall_score(y_true, predictions, zero_division=0),
            "f1": f1_score(y_true, predictions, zero_division=0),
        })
    search = pd.DataFrame(rows)
    selected_threshold = float(
        search.sort_values(["f2", "recall"], ascending=False).iloc[0]["threshold"]
    )
    return selected_threshold, search

def calculate_ids_metrics(
    condition: str,
    seed: int,
    run_source: str,
    formal_dp: bool,
    target_epsilon: float | None,
    actual_epsilon: float | None,
    split_name: str,
    threshold_policy: str,
    y_true: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> dict:
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "condition": condition,
        "seed": seed,
        "run_source": run_source,
        "formal_dp": formal_dp,
        "target_epsilon": target_epsilon,
        "actual_epsilon": actual_epsilon,
        "delta": TARGET_DELTA if formal_dp else np.nan,
        "split": split_name,
        "threshold_policy": threshold_policy,
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "fnr": fn / (fn + tp) if fn + tp else np.nan,
        "fpr": fp / (fp + tn) if fp + tn else np.nan,
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def build_target_mia_features(
    condition: str,
    seed: int,
    train_probabilities: np.ndarray,
    validation_probabilities: np.ndarray,
) -> pd.DataFrame:
    probability_lookup = pd.concat([
        pd.DataFrame({
            "membership": 1,
            "partition_position": target_member_positions,
            "prob_attack": train_probabilities[target_member_positions],
        }),
        pd.DataFrame({
            "membership": 0,
            "partition_position": target_nonmember_positions,
            "prob_attack": validation_probabilities[target_nonmember_positions],
        }),
    ], ignore_index=True)

    merged = target_mia_sample_manifest.merge(
        probability_lookup,
        on=["membership", "partition_position"],
        how="left",
        validate="one_to_one",
    )
    assert merged["prob_attack"].notna().all()

    true_labels = merged["true_label"].to_numpy(dtype=int)
    prob_attack = merged["prob_attack"].to_numpy()
    true_class_probability = np.where(true_labels == 1, prob_attack, 1 - prob_attack)
    true_class_probability = np.clip(true_class_probability, 1e-12, 1 - 1e-12)

    merged.insert(0, "seed", seed)
    merged.insert(0, "condition", condition)
    merged["confidence"] = np.maximum(prob_attack, 1 - prob_attack)
    merged["loss"] = -np.log(true_class_probability)
    merged["correctness"] = (
        (prob_attack >= 0.5).astype(int) == true_labels
    ).astype(int)
    return merged


## 6. Strict cache signatures and accepted seed-42 import


In [28]:
@dataclass
class TargetRunResult:
    condition: str
    seed: int
    formal_dp: bool
    target_epsilon: float | None
    actual_epsilon: float | None
    delta: float | None
    noise_multiplier: float | None
    max_grad_norm: float | None
    sample_rate: float | None
    ids_results: pd.DataFrame
    target_mia_features: pd.DataFrame
    config: dict
    warning_rows: list[dict]

def run_paths(condition: str, seed: int) -> dict[str, Path]:
    run_dir = INTERMEDIATE_DIR / condition / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return {
        "dir": run_dir,
        "state": MODEL_DIR / f"{condition}_seed_{seed}.pt",
        "config": run_dir / "target_config.json",
        "ids": run_dir / "target_ids_results.csv",
        "mia": run_dir / "target_mia_features.csv",
        "history": run_dir / "target_training_history.csv",
        "threshold": run_dir / "target_threshold_search.csv",
        "warnings": run_dir / "target_warnings.csv",
    }

def expected_run_spec(condition_spec: dict, seed: int) -> dict:
    return {
        "protocol_version": PROTOCOL_VERSION,
        "protocol_revision": PROTOCOL_REVISION,
        "condition": condition_spec["condition"],
        "seed": seed,
        "formal_dp": condition_spec["formal_dp"],
        "target_epsilon": condition_spec["target_epsilon"],
        "target_delta": TARGET_DELTA if condition_spec["formal_dp"] else None,
        "architecture": {
            "input_dim": TARGET_INPUT_DIM,
            "hidden_dims": list(HIDDEN_DIMS),
            "activation": "ReLU",
            "output": "one logit",
            "batch_norm": False,
        },
        "training": {
            "optimizer": "Adam",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "loss": "BCEWithLogitsLoss",
        },
        "dp": {
            "max_grad_norm": MAX_GRAD_NORM if condition_spec["formal_dp"] else None,
            "accountant": ACCOUNTANT if condition_spec["formal_dp"] else None,
            "poisson_sampling": POISSON_SAMPLING if condition_spec["formal_dp"] else False,
            "secure_mode": SECURE_MODE,
        },
        "dataset_hashes": {
            "train_sha256": actual_train_hash,
            "test_sha256": actual_test_hash,
        },
        "split_hashes": split_hashes,
        "preprocessor_sha256": preprocessor_hash,
        "target_mia_sample_sha256": target_sample_hash,
        "runtime": RUNTIME_IDENTITY,
    }

def validate_result_frames(
    condition: str,
    seed: int,
    ids_results: pd.DataFrame,
    target_mia_features: pd.DataFrame,
) -> None:
    assert len(ids_results) == 3
    assert set(ids_results["condition"]) == {condition}
    assert set(ids_results["seed"].astype(int)) == {seed}
    assert ids_results[["recall", "fnr", "f1", "pr_auc"]].notna().all().all()

    assert len(target_mia_features) == len(target_mia_sample_manifest)
    assert set(target_mia_features["condition"]) == {condition}
    assert set(target_mia_features["seed"].astype(int)) == {seed}
    keys = ["membership", "partition_position", "row_id"]
    pd.testing.assert_frame_equal(
        target_mia_features[keys].reset_index(drop=True),
        target_mia_sample_manifest[keys].reset_index(drop=True),
        check_dtype=False,
    )
    assert target_mia_features[
        ["prob_attack", "confidence", "loss", "correctness"]
    ].notna().all().all()

def load_cached_run(condition_spec: dict, seed: int) -> TargetRunResult | None:
    paths = run_paths(condition_spec["condition"], seed)
    if not all(paths[name].exists() for name in ["config", "ids", "mia"]):
        return None

    saved_config = load_json(paths["config"])
    expected_fingerprint = canonical_hash(expected_run_spec(condition_spec, seed))
    if saved_config.get("cache_fingerprint") != expected_fingerprint:
        print(f"Ignoring stale cache: {condition_spec['condition']}, seed {seed}")
        return None

    assert saved_config.get("run_source") in {
        "accepted_experiment_05",
        "trained_experiment_06",
    }
    if saved_config["run_source"] == "accepted_experiment_05":
        assert seed == EXPERIMENT_05_SEED
        assert saved_config.get(
            "accepted_experiment_05_manifest_sha256"
        ) == calculate_sha256(EXP05_MANIFEST_FILE)

    if saved_config["run_source"] == "trained_experiment_06":
        model_path = Path(saved_config["model_state_path"])
        if (
            not model_path.exists()
            or calculate_sha256(model_path) != saved_config["model_sha256"]
        ):
            print(f"Ignoring incomplete model cache: {condition_spec['condition']}, seed {seed}")
            return None

    output_hashes = saved_config.get("output_sha256", {})
    if any(output_hashes.get(name) != calculate_sha256(paths[name]) for name in ["ids", "mia"]):
        print(f"Ignoring changed result cache: {condition_spec['condition']}, seed {seed}")
        return None

    ids_results = pd.read_csv(paths["ids"])
    target_mia_features = pd.read_csv(paths["mia"])
    validate_result_frames(
        condition_spec["condition"],
        seed,
        ids_results,
        target_mia_features,
    )

    warning_rows = (
        pd.read_csv(paths["warnings"]).to_dict(orient="records")
        if paths["warnings"].exists()
        else []
    )
    print(f"Resuming validated cache: {condition_spec['condition']}, seed {seed}")
    return TargetRunResult(
        condition=condition_spec["condition"],
        seed=seed,
        formal_dp=condition_spec["formal_dp"],
        target_epsilon=condition_spec["target_epsilon"],
        actual_epsilon=saved_config.get("actual_epsilon"),
        delta=saved_config.get("delta"),
        noise_multiplier=saved_config.get("noise_multiplier"),
        max_grad_norm=saved_config.get("max_grad_norm"),
        sample_rate=saved_config.get("sample_rate"),
        ids_results=ids_results,
        target_mia_features=target_mia_features,
        config=saved_config,
        warning_rows=warning_rows,
    )


In [29]:
exp05_ids = pd.read_csv(EXP05_IDS_FILE)
exp05_mia = pd.read_csv(EXP05_MIA_FILE)
exp05_target_configs = pd.read_csv(EXP05_TARGET_CONFIGS_FILE)

def import_accepted_seed_42(condition_spec: dict) -> TargetRunResult:
    condition = condition_spec["condition"]
    seed = EXPERIMENT_05_SEED
    paths = run_paths(condition, seed)

    config_matches = exp05_target_configs[
        exp05_target_configs["condition"] == condition
    ]
    assert len(config_matches) == 1
    accepted_config = config_matches.iloc[0]

    assert bool(accepted_config["formal_dp"]) == condition_spec["formal_dp"]
    if condition_spec["formal_dp"]:
        assert np.isclose(
            float(accepted_config["target_epsilon"]),
            float(condition_spec["target_epsilon"]),
        )
        assert abs(
            float(accepted_config["actual_epsilon"])
            - float(condition_spec["target_epsilon"])
        ) <= 0.10
        assert np.isclose(
            float(accepted_config["delta"]),
            TARGET_DELTA,
            rtol=0,
            atol=1e-15,
        )

    ids_results = exp05_ids[exp05_ids["condition"] == condition].copy()
    assert len(ids_results) == 3
    ids_results.insert(1, "seed", seed)
    ids_results.insert(2, "run_source", "accepted_experiment_05")
    if "target_epsilon" not in ids_results.columns:
        ids_results.insert(
            4,
            "target_epsilon",
            condition_spec["target_epsilon"],
        )

    accepted_mia_path = (
        EXP05_INTERMEDIATE_DIR / condition / "target_mia_features.csv"
    )
    if accepted_mia_path.exists():
        target_mia_features = pd.read_csv(accepted_mia_path)
        if "seed" not in target_mia_features.columns:
            target_mia_features.insert(1, "seed", seed)
        print(f"Importing accepted Experiment 05 target features: {condition}")
    else:
        model_path = Path(str(accepted_config["model_state_path"]))
        if not model_path.exists():
            raise FileNotFoundError(
                f"Accepted seed-42 feature cache and model are both missing for {condition}. Recover the original Experiment 05 files; no replacement training was started."
            )
        assert calculate_sha256(model_path) == accepted_config["model_sha256"]
        model = BinaryMLP(TARGET_INPUT_DIM).to(DEVICE)
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        train_probabilities = predict_probabilities(
            model, X_target_train, y_target_train, DEVICE
        )
        validation_probabilities = predict_probabilities(
            model, X_target_validation, y_target_validation, DEVICE
        )
        target_mia_features = build_target_mia_features(
            condition,
            seed,
            train_probabilities,
            validation_probabilities,
        )
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"Reconstructed accepted Experiment 05 target features: {condition}")

    if "seed" not in target_mia_features.columns:
        target_mia_features.insert(1, "seed", seed)

    fingerprint = canonical_hash(expected_run_spec(condition_spec, seed))
    imported_config = {
        **expected_run_spec(condition_spec, seed),
        "cache_fingerprint": fingerprint,
        "run_source": "accepted_experiment_05",
        "actual_epsilon": (
            float(accepted_config["actual_epsilon"])
            if condition_spec["formal_dp"]
            else None
        ),
        "delta": TARGET_DELTA if condition_spec["formal_dp"] else None,
        "noise_multiplier": (
            float(accepted_config["noise_multiplier"])
            if condition_spec["formal_dp"]
            else None
        ),
        "max_grad_norm": (
            float(accepted_config["max_grad_norm"])
            if condition_spec["formal_dp"]
            else None
        ),
        "sample_rate": (
            float(accepted_config["sample_rate"])
            if condition_spec["formal_dp"]
            else None
        ),
        "training_seconds": float(accepted_config["training_seconds"]),
        "selected_threshold": float(accepted_config["selected_threshold"]),
        "model_state_path": str(accepted_config["model_state_path"]),
        "model_sha256": str(accepted_config["model_sha256"]),
        "accepted_ids_sha256": calculate_sha256(EXP05_IDS_FILE),
        "source_software": exp05_manifest["software"],
        "source_device": exp05_manifest["device"],
        "accepted_experiment_05_manifest_sha256": calculate_sha256(
            EXP05_MANIFEST_FILE
        ),
    }

    accepted_warnings = pd.read_csv(EXP05_WARNING_FILE)
    accepted_warnings = accepted_warnings[
        (accepted_warnings["condition"] == condition)
        & accepted_warnings["stage"].str.startswith("target_")
    ].copy()
    accepted_warnings["seed"] = seed
    warning_rows = accepted_warnings[["stage", "condition", "seed", "category", "message", "occurrences"]].to_dict(orient="records")
    pd.DataFrame(warning_rows, columns=["stage", "condition", "seed", "category", "message", "occurrences"]).to_csv(paths["warnings"], index=False)
    validate_result_frames(condition, seed, ids_results, target_mia_features)
    ids_results.to_csv(paths["ids"], index=False)
    target_mia_features.to_csv(paths["mia"], index=False)
    imported_config["output_sha256"] = {name: calculate_sha256(paths[name]) for name in ["ids", "mia"]}
    write_strict_json(paths["config"], imported_config)

    return TargetRunResult(
        condition=condition,
        seed=seed,
        formal_dp=condition_spec["formal_dp"],
        target_epsilon=condition_spec["target_epsilon"],
        actual_epsilon=imported_config["actual_epsilon"],
        delta=imported_config["delta"],
        noise_multiplier=imported_config["noise_multiplier"],
        max_grad_norm=imported_config["max_grad_norm"],
        sample_rate=imported_config["sample_rate"],
        ids_results=ids_results,
        target_mia_features=target_mia_features,
        config=imported_config,
        warning_rows=warning_rows,
    )


In [30]:
def train_target_run(condition_spec: dict, seed: int) -> TargetRunResult:
    cached = load_cached_run(condition_spec, seed) if RESUME else None
    if cached is not None:
        return cached

    if seed == EXPERIMENT_05_SEED and REUSE_ACCEPTED_SEED_42:
        return import_accepted_seed_42(condition_spec)

    condition = condition_spec["condition"]
    formal_dp = condition_spec["formal_dp"]
    target_epsilon = condition_spec["target_epsilon"]
    paths = run_paths(condition, seed)
    run_source = "trained_experiment_06"

    set_all_seeds(seed)
    model = BinaryMLP(TARGET_INPUT_DIM)
    model.load_state_dict(create_initial_state(TARGET_INPUT_DIM, seed))
    model = model.to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    criterion = nn.BCEWithLogitsLoss()
    train_loader = create_train_loader(X_target_train, y_target_train, seed)

    privacy_engine = None
    actual_epsilon = None
    noise_multiplier = None
    actual_max_grad_norm = None
    sample_rate = None
    warning_rows = []

    if formal_dp:
        privacy_engine = PrivacyEngine(accountant=ACCOUNTANT, secure_mode=SECURE_MODE)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("once")
            model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
                module=model,
                optimizer=optimizer,
                criterion=criterion,
                data_loader=train_loader,
                target_epsilon=target_epsilon,
                target_delta=TARGET_DELTA,
                epochs=EPOCHS,
                max_grad_norm=MAX_GRAD_NORM,
                poisson_sampling=POISSON_SAMPLING,
                clipping="flat",
                loss_reduction="mean",
            )
        warning_rows.extend(
            summarize_warnings(caught, "target_make_private", condition, seed)
        )
        noise_multiplier = float(optimizer.noise_multiplier)
        actual_max_grad_norm = float(optimizer.max_grad_norm)
        sample_rate = float(
            getattr(train_loader, "sample_rate", 1.0 / len(train_loader))
        )

    history_rows = []
    started_at = time.time()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("once")
        for epoch in range(1, EPOCHS + 1):
            epoch_loss = train_one_epoch(
                model,
                train_loader,
                optimizer,
                criterion,
                DEVICE,
            )
            epsilon_after_epoch = (
                float(privacy_engine.get_epsilon(TARGET_DELTA))
                if formal_dp
                else np.nan
            )
            history_rows.append({
                "condition": condition,
                "seed": seed,
                "epoch": epoch,
                "train_loss": epoch_loss,
                "epsilon": epsilon_after_epoch,
                "delta": TARGET_DELTA if formal_dp else np.nan,
            })
            if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
                epsilon_text = (
                    f", epsilon={epsilon_after_epoch:.4f}" if formal_dp else ""
                )
                print(
                    f"{condition}, seed {seed}: epoch {epoch:02d}/{EPOCHS}, "
                    f"loss={epoch_loss:.6f}{epsilon_text}"
                )
    warning_rows.extend(
        summarize_warnings(caught, "target_training", condition, seed)
    )
    training_seconds = time.time() - started_at

    if formal_dp:
        actual_epsilon = float(privacy_engine.get_epsilon(TARGET_DELTA))
        assert np.isfinite(actual_epsilon)
        assert abs(actual_epsilon - target_epsilon) <= 0.10

    train_probabilities = predict_probabilities(
        model, X_target_train, y_target_train, DEVICE
    )
    validation_probabilities = predict_probabilities(
        model, X_target_validation, y_target_validation, DEVICE
    )
    test_probabilities = predict_probabilities(model, X_test, y_test, DEVICE)

    selected_threshold, threshold_search = select_f2_threshold(
        y_target_validation.reshape(-1).astype(int),
        validation_probabilities,
    )

    ids_results = pd.DataFrame([
        calculate_ids_metrics(
            condition, seed, run_source, formal_dp, target_epsilon,
            actual_epsilon, "target_validation", "default_0_5",
            y_target_validation.reshape(-1).astype(int),
            validation_probabilities, 0.5,
        ),
        calculate_ids_metrics(
            condition, seed, run_source, formal_dp, target_epsilon,
            actual_epsilon, "KDDTest+", "default_0_5",
            y_test.reshape(-1).astype(int), test_probabilities, 0.5,
        ),
        calculate_ids_metrics(
            condition, seed, run_source, formal_dp, target_epsilon,
            actual_epsilon, "KDDTest+", "validation_selected_F2",
            y_test.reshape(-1).astype(int), test_probabilities,
            selected_threshold,
        ),
    ])

    target_mia_features = build_target_mia_features(
        condition,
        seed,
        train_probabilities,
        validation_probabilities,
    )

    torch.save(unwrap_model(model).state_dict(), paths["state"])
    model_sha256 = calculate_sha256(paths["state"])
    ids_results.to_csv(paths["ids"], index=False)
    target_mia_features.to_csv(paths["mia"], index=False)
    pd.DataFrame(history_rows).to_csv(paths["history"], index=False)
    threshold_search.to_csv(paths["threshold"], index=False)
    pd.DataFrame(
        warning_rows,
        columns=["stage", "condition", "seed", "category", "message", "occurrences"],
    ).to_csv(paths["warnings"], index=False)

    target_config = {
        **expected_run_spec(condition_spec, seed),
        "cache_fingerprint": canonical_hash(expected_run_spec(condition_spec, seed)),
        "run_source": run_source,
        "actual_epsilon": actual_epsilon,
        "delta": TARGET_DELTA if formal_dp else None,
        "noise_multiplier": noise_multiplier,
        "max_grad_norm": actual_max_grad_norm,
        "sample_rate": sample_rate,
        "training_seconds": training_seconds,
        "selected_threshold": selected_threshold,
        "model_state_path": str(paths["state"]),
        "model_sha256": model_sha256,
    }
    target_config["output_sha256"] = {name: calculate_sha256(paths[name]) for name in ["ids", "mia"]}
    write_strict_json(paths["config"], target_config)

    validate_result_frames(condition, seed, ids_results, target_mia_features)

    del model, optimizer, train_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return TargetRunResult(
        condition=condition,
        seed=seed,
        formal_dp=formal_dp,
        target_epsilon=target_epsilon,
        actual_epsilon=actual_epsilon,
        delta=TARGET_DELTA if formal_dp else None,
        noise_multiplier=noise_multiplier,
        max_grad_norm=actual_max_grad_norm,
        sample_rate=sample_rate,
        ids_results=ids_results,
        target_mia_features=target_mia_features,
        config=target_config,
        warning_rows=warning_rows,
    )


## 7. Reconstruct and freeze primary and secondary attackers before training

Primary families are fixed. Secondary selection uses only accepted Experiment 05 shadow
calibration AUC, with a deterministic tie-break. No target results select attackers.


In [31]:
def calculate_mia_advantage(
    membership_labels: np.ndarray,
    membership_scores: np.ndarray,
) -> float:
    false_positive_rates, true_positive_rates, _ = roc_curve(
        membership_labels,
        membership_scores,
    )
    return float(np.max(true_positive_rates - false_positive_rates))

def calculate_tpr_at_fpr(
    membership_labels: np.ndarray,
    membership_scores: np.ndarray,
    maximum_fpr: float,
) -> float:
    false_positive_rates, true_positive_rates, _ = roc_curve(
        membership_labels,
        membership_scores,
    )
    eligible = false_positive_rates <= maximum_fpr
    return float(np.max(true_positive_rates[eligible])) if np.any(eligible) else 0.0

def find_balanced_accuracy_threshold(
    membership_labels: np.ndarray,
    membership_scores: np.ndarray,
) -> tuple[float, float]:
    candidate_thresholds = np.unique(
        np.quantile(membership_scores, np.linspace(0.001, 0.999, 400))
    )
    best_threshold = float(candidate_thresholds[0])
    best_balanced_accuracy = -1.0
    for threshold in candidate_thresholds:
        predictions = membership_scores >= threshold
        balanced_accuracy = balanced_accuracy_score(
            membership_labels,
            predictions,
        )
        if balanced_accuracy > best_balanced_accuracy:
            best_threshold = float(threshold)
            best_balanced_accuracy = float(balanced_accuracy)
    return best_threshold, best_balanced_accuracy

@dataclass
class CalibratedAttack:
    condition: str
    threat_model: str
    attack_model: str
    feature_set: str
    operating_threshold: float
    shadow_calibration_auc: float
    score_function: Callable[[pd.DataFrame], np.ndarray]

def build_selected_attack(
    selected_row: pd.Series,
    shadow_data: pd.DataFrame,
) -> CalibratedAttack:
    condition = str(selected_row["condition"])
    threat_model = str(selected_row["threat_model"])
    attack_model = str(selected_row["attack_model"])

    attack_train = shadow_data[
        shadow_data["shadow_seed"] != CALIBRATION_SHADOW_SEED
    ].copy()
    attack_calibration = shadow_data[
        shadow_data["shadow_seed"] == CALIBRATION_SHADOW_SEED
    ].copy()

    if attack_model in {"confidence_threshold", "loss_threshold"}:
        feature_column = (
            "confidence" if attack_model == "confidence_threshold" else "loss"
        )
        score_direction = 1.0 if attack_model == "confidence_threshold" else -1.0
        calibration_scores = (
            score_direction * attack_calibration[feature_column].to_numpy()
        )
        operating_threshold, _ = find_balanced_accuracy_threshold(
            attack_calibration["membership"].to_numpy(dtype=int),
            calibration_scores,
        )
        score_function = lambda dataframe, column=feature_column, direction=score_direction: (
            direction * dataframe[column].to_numpy()
        )
        feature_set = feature_column
    else:
        feature_columns = (
            ["prob_attack"]
            if threat_model == "score_only_black_box"
            else ["loss", "correctness"]
        )
        if attack_model == "logistic_regression":
            model = Pipeline([
                ("scale", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                        random_state=BASE_SEED,
                    ),
                ),
            ])
        elif attack_model == "random_forest":
            model = RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=5,
                class_weight="balanced_subsample",
                random_state=BASE_SEED,
                n_jobs=-1,
            )
        else:
            raise ValueError(f"Unsupported selected attacker: {attack_model}")

        model.fit(
            attack_train[feature_columns],
            attack_train["membership"],
        )
        calibration_scores = model.predict_proba(
            attack_calibration[feature_columns]
        )[:, 1]
        operating_threshold, _ = find_balanced_accuracy_threshold(
            attack_calibration["membership"].to_numpy(dtype=int),
            calibration_scores,
        )
        score_function = lambda dataframe, fitted=model, columns=feature_columns: (
            fitted.predict_proba(dataframe[columns])[:, 1]
        )
        feature_set = " + ".join(feature_columns)

    calibration_auc = roc_auc_score(
        attack_calibration["membership"].to_numpy(dtype=int),
        calibration_scores,
    )
    return CalibratedAttack(
        condition=condition,
        threat_model=threat_model,
        attack_model=attack_model,
        feature_set=feature_set,
        operating_threshold=float(operating_threshold),
        shadow_calibration_auc=float(calibration_auc),
        score_function=score_function,
    )


In [32]:
accepted_calibration = pd.read_csv(EXP05_ATTACK_CALIBRATION_FILE)
relevant_conditions = {spec["condition"] for spec in CONDITIONS}
accepted_calibration = accepted_calibration[
    accepted_calibration["condition"].isin(relevant_conditions)
].copy()

secondary_calibration_rows = (
    accepted_calibration.assign(_original_order=np.arange(len(accepted_calibration)))
    .sort_values(
        ["condition", "threat_model", "shadow_calibration_auc", "_original_order"],
        ascending=[True, True, False, True],
    )
    .groupby(["condition", "threat_model"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
assert len(secondary_calibration_rows) == len(CONDITIONS) * 2
primary_calibration_rows = accepted_calibration[
    accepted_calibration.apply(
        lambda row: row["attack_model"] == PRIMARY_ATTACK_MODELS[row["threat_model"]],
        axis=1,
    )
].copy()
assert len(primary_calibration_rows) == len(CONDITIONS) * 2
selected_calibration_rows = pd.concat([
    primary_calibration_rows.assign(analysis_role="primary"),
    secondary_calibration_rows.assign(analysis_role="secondary"),
], ignore_index=True)

fixed_attacks = {}
secondary_attacks = {}
reconstructed_attacks = {}
shadow_data_hashes = {}
fixed_attacker_verification_rows = []

for condition in sorted(relevant_conditions):
    shadow_path = (
        EXP05_INTERMEDIATE_DIR / condition / "shadow_attack_data.csv"
    )
    shadow_data_hashes[condition] = calculate_sha256(shadow_path)
    shadow_data = pd.read_csv(shadow_path)
    assert set(shadow_data["condition"]) == {condition}
    assert set(shadow_data["shadow_seed"].astype(int)) == set(SHADOW_SEEDS)
    assert set(shadow_data["membership"].astype(int)) == {0, 1}

    condition_rows = selected_calibration_rows[
        selected_calibration_rows["condition"] == condition
    ]
    for _, selected_row in condition_rows.iterrows():
        attack_key = (condition, selected_row["threat_model"], selected_row["attack_model"])
        if attack_key not in reconstructed_attacks:
            reconstructed_attacks[attack_key] = build_selected_attack(selected_row, shadow_data)
        attack = reconstructed_attacks[attack_key]
        threshold_matches = np.isclose(
            attack.operating_threshold,
            float(selected_row["operating_threshold"]),
            rtol=0,
            atol=1e-8,
        )
        auc_matches = np.isclose(
            attack.shadow_calibration_auc,
            float(selected_row["shadow_calibration_auc"]),
            rtol=0,
            atol=1e-10,
        )
        passed = bool(threshold_matches and auc_matches)
        destination = fixed_attacks if selected_row["analysis_role"] == "primary" else secondary_attacks
        destination[(condition, attack.threat_model)] = attack
        fixed_attacker_verification_rows.append({
            "analysis_role": selected_row["analysis_role"],
            "condition": condition,
            "threat_model": attack.threat_model,
            "attack_model": attack.attack_model,
            "feature_set": attack.feature_set,
            "accepted_operating_threshold": selected_row["operating_threshold"],
            "reconstructed_operating_threshold": attack.operating_threshold,
            "accepted_shadow_calibration_auc": selected_row["shadow_calibration_auc"],
            "reconstructed_shadow_calibration_auc": attack.shadow_calibration_auc,
            "verification_passed": passed,
        })

fixed_attacker_verification = pd.DataFrame(
    fixed_attacker_verification_rows
)
fixed_attacker_verification.to_csv(
    RESULTS_DIR / "fixed_attacker_verification.csv",
    index=False,
)
assert fixed_attacker_verification["verification_passed"].all()
display(fixed_attacker_verification)


,analysis_role,condition,threat_model,attack_model,feature_set,accepted_operating_threshold,reconstructed_operating_threshold,accepted_shadow_calibration_auc,reconstructed_shadow_calibration_auc,verification_passed
0,primary,dp_eps_2,score_only_black_box,logistic_regression,prob_attack,0.499802,0.499802,0.501878,0.501878,True
1,primary,dp_eps_2,label_aware_audit,loss_threshold,loss,-0.000017,-0.000017,0.497049,0.497049,True
2,secondary,dp_eps_2,label_aware_audit,loss_threshold,loss,-0.000017,-0.000017,0.497049,0.497049,True
3,secondary,dp_eps_2,score_only_black_box,logistic_regression,prob_attack,0.499802,0.499802,0.501878,0.501878,True
4,primary,dp_eps_4,score_only_black_box,logistic_regression,prob_attack,0.499856,0.499856,0.501754,0.501754,True
5,primary,dp_eps_4,label_aware_audit,loss_threshold,loss,-0.000008,-0.000008,0.498117,0.498117,True
6,secondary,dp_eps_4,label_aware_audit,random_forest,loss + correctness,0.558856,0.558856,0.502600,0.502600,True
7,secondary,dp_eps_4,score_only_black_box,random_forest,prob_attack,0.658049,0.658049,0.502794,0.502794,True
8,primary,non_private,score_only_black_box,logistic_regression,prob_attack,0.499860,0.499860,0.500934,0.500934,True
9,primary,non_private,label_aware_audit,loss_threshold,loss,-0.000015,-0.000015,0.499789,0.499789,True


## 8. Verify seed 42 before new training


In [33]:
def evaluate_fixed_attack(
    attack: CalibratedAttack,
    evaluation_data: pd.DataFrame,
) -> dict:
    membership_labels = evaluation_data["membership"].to_numpy(dtype=int)
    membership_scores = attack.score_function(evaluation_data)
    membership_predictions = (
        membership_scores >= attack.operating_threshold
    ).astype(int)
    return {
        "condition": attack.condition,
        "seed": int(evaluation_data["seed"].iloc[0]),
        "subset": "overall",
        "threat_model": attack.threat_model,
        "attack_model": attack.attack_model,
        "feature_set": attack.feature_set,
        "n": len(evaluation_data),
        "members": int(membership_labels.sum()),
        "nonmembers": int((1 - membership_labels).sum()),
        "mia_auc": roc_auc_score(membership_labels, membership_scores),
        "mia_advantage": calculate_mia_advantage(
            membership_labels,
            membership_scores,
        ),
        "mia_balanced_accuracy": balanced_accuracy_score(
            membership_labels,
            membership_predictions,
        ),
        "mia_precision": precision_score(
            membership_labels,
            membership_predictions,
            zero_division=0,
        ),
        "mia_recall": recall_score(
            membership_labels,
            membership_predictions,
            zero_division=0,
        ),
        "tpr_at_1pct_fpr": calculate_tpr_at_fpr(
            membership_labels,
            membership_scores,
            0.01,
        ),
        "tpr_at_5pct_fpr": calculate_tpr_at_fpr(
            membership_labels,
            membership_scores,
            0.05,
        ),
        "operating_threshold": attack.operating_threshold,
        "shadow_calibration_auc": attack.shadow_calibration_auc,
    }


def evaluate_policy(runs, attacks, role):
    rows = []
    for (condition, seed), run in runs.items():
        for threat_model in PRIMARY_ATTACK_MODELS:
            row = evaluate_fixed_attack(attacks[(condition, threat_model)], run.target_mia_features)
            row["analysis_role"] = role
            rows.append(row)
    return pd.DataFrame(rows)

PREFLIGHT_PASSED = False
seed42_runs = {}
for spec in CONDITIONS:
    result = train_target_run(spec, EXPERIMENT_05_SEED)
    assert result.config["run_source"] == "accepted_experiment_05"
    seed42_runs[(result.condition, result.seed)] = result
seed42_mia = pd.concat([
    evaluate_policy(seed42_runs, fixed_attacks, "primary"),
    evaluate_policy(seed42_runs, secondary_attacks, "secondary"),
], ignore_index=True)
seed42_verification_rows = []
for _, repeated_row in seed42_mia.iterrows():
    accepted_match = exp05_mia[
        (exp05_mia["condition"] == repeated_row["condition"])
        & (exp05_mia["subset"] == "overall")
        & (exp05_mia["threat_model"] == repeated_row["threat_model"])
        & (exp05_mia["attack_model"] == repeated_row["attack_model"])
    ]
    assert len(accepted_match) == 1
    accepted_row = accepted_match.iloc[0]
    for metric in ["mia_auc", "mia_advantage"]:
        accepted_value = float(accepted_row[metric])
        repeated_value = float(repeated_row[metric])
        # Numerical reproduction policy, not a change to accepted experiment results.
        # Observed discrepancy: 9.45e-9 in one LR AUC (duplicated across roles).
        # At 12,597 x 12,597 member/nonmember pairs this is ~1.5 pair units.
        # Exact cause is unconfirmed; CSV round-tripping/near-ties are plausible.
        # Allow at most 1e-8 absolute AUC difference; advantage stays at 1e-10.
        verification_atol = 1e-8 if metric == "mia_auc" else 1e-10
        seed42_verification_rows.append({
            "analysis_role": repeated_row["analysis_role"],
            "condition": repeated_row["condition"],
            "threat_model": repeated_row["threat_model"],
            "attack_model": repeated_row["attack_model"],
            "metric": metric,
            "accepted_experiment_05_value": accepted_value,
            "recomputed_value": repeated_value,
            "absolute_difference": abs(repeated_value - accepted_value),
            "verification_atol": verification_atol,
            "verification_policy": "auc_absolute_1e-8_advantage_absolute_1e-10_v1",
            "verification_passed": bool(
                np.isclose(repeated_value, accepted_value, rtol=0, atol=verification_atol)
            ),
        })

seed42_import_verification = pd.DataFrame(seed42_verification_rows)
seed42_import_verification.to_csv(
    RESULTS_DIR / "seed42_import_verification.csv",
    index=False,
)
failed_verifications = seed42_import_verification.loc[
    ~seed42_import_verification["verification_passed"]
].copy()
if not failed_verifications.empty:
    print("\nSeed-42 verification differences (accepted vs recomputed):")
    print(failed_verifications.to_string(
        index=False, float_format=lambda value: f"{value:.17g}"
    ))
    print("\nImported target-feature dtypes:")
    for (condition, seed), run in seed42_runs.items():
        print(condition, run.target_mia_features[
            ["prob_attack", "confidence", "loss", "correctness"]
        ].dtypes.astype(str).to_dict())
    print("\nVerification CSV:", RESULTS_DIR / "seed42_import_verification.csv")
    raise RuntimeError(
        "Seed-42 metrics do not reproduce the accepted Experiment 05 results. "
        "No new target training has started. Share the table above or "
        "seed42_import_verification.csv; do not loosen the tolerance or delete caches."
    )
assert seed42_import_verification["verification_passed"].all()


PREFLIGHT_PASSED = bool(
    fixed_attacker_verification["verification_passed"].all()
    and seed42_import_verification["verification_passed"].all()
)
assert PREFLIGHT_PASSED
print("Experiment 06 preflight: PASSED. Accepted seed 42 and frozen attackers verified. "
      "Proceeding to at most 12 new target trainings (validated caches will resume).")


Resuming validated cache: non_private, seed 42
Resuming validated cache: dp_eps_4, seed 42
Resuming validated cache: dp_eps_2, seed 42
Experiment 06 preflight: PASSED. Accepted seed 42 and frozen attackers verified. Proceeding to at most 12 new target trainings (validated caches will resume).


## 9. Train or resume the 15 condition-seed target runs


In [34]:
assert PREFLIGHT_PASSED
# Accepted seed 42 has already been imported and checked.
target_runs = dict(seed42_runs)

for seed in RUN_SEEDS:
    for condition_spec in CONDITIONS:
        if seed == EXPERIMENT_05_SEED:
            continue
        result = train_target_run(condition_spec, seed)
        target_runs[(result.condition, result.seed)] = result

repeated_run_ids_results = pd.concat(
    [result.ids_results for result in target_runs.values()],
    ignore_index=True,
)
repeated_run_configs = pd.DataFrame([
    {
        "condition": result.condition,
        "seed": result.seed,
        "run_source": result.config["run_source"],
        "formal_dp": result.formal_dp,
        "target_epsilon": result.target_epsilon,
        "actual_epsilon": result.actual_epsilon,
        "delta": result.delta,
        "noise_multiplier": result.noise_multiplier,
        "max_grad_norm": result.max_grad_norm,
        "batch_size": BATCH_SIZE,
        "sample_rate": result.sample_rate,
        "epochs": EPOCHS,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "accountant": ACCOUNTANT if result.formal_dp else None,
        "poisson_sampling": POISSON_SAMPLING if result.formal_dp else False,
        "secure_mode": SECURE_MODE,
        "training_seconds": result.config["training_seconds"],
        "selected_threshold": result.config["selected_threshold"],
        "model_state_path": result.config["model_state_path"],
        "model_sha256": result.config["model_sha256"],
        "cache_fingerprint": result.config["cache_fingerprint"],
        "output_sha256": result.config["output_sha256"],
        "source_device": result.config.get("source_device", str(DEVICE)),
        "source_software": result.config.get("source_software", RUNTIME_IDENTITY),
    }
    for result in target_runs.values()
])

repeated_run_ids_results.to_csv(
    RESULTS_DIR / "repeated_run_ids_results.csv",
    index=False,
)
repeated_run_configs.to_csv(
    RESULTS_DIR / "repeated_run_configs.csv",
    index=False,
)

display(
    repeated_run_ids_results[
        (repeated_run_ids_results["split"] == "KDDTest+")
        & (
            repeated_run_ids_results["threshold_policy"]
            == "validation_selected_F2"
        )
    ][[
        "condition", "seed", "actual_epsilon", "threshold",
        "recall", "fnr", "f1", "fpr", "pr_auc",
    ]]
)


Ignoring stale cache: non_private, seed 52
non_private, seed 52: epoch 01/30, loss=0.167820
non_private, seed 52: epoch 05/30, loss=0.041732
non_private, seed 52: epoch 10/30, loss=0.029388
non_private, seed 52: epoch 15/30, loss=0.022029
non_private, seed 52: epoch 20/30, loss=0.017848
non_private, seed 52: epoch 25/30, loss=0.015799
non_private, seed 52: epoch 30/30, loss=0.014673
Ignoring stale cache: dp_eps_4, seed 52


/usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


dp_eps_4, seed 52: epoch 01/30, loss=0.259699, epsilon=1.3784
dp_eps_4, seed 52: epoch 05/30, loss=0.117820, epsilon=2.0538
dp_eps_4, seed 52: epoch 10/30, loss=0.098541, epsilon=2.5569
dp_eps_4, seed 52: epoch 15/30, loss=0.094164, epsilon=2.9719
dp_eps_4, seed 52: epoch 20/30, loss=0.087911, epsilon=3.3417
dp_eps_4, seed 52: epoch 25/30, loss=0.082083, epsilon=3.6813
dp_eps_4, seed 52: epoch 30/30, loss=0.078342, epsilon=3.9983
Ignoring stale cache: dp_eps_2, seed 52


/usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


dp_eps_2, seed 52: epoch 01/30, loss=0.274070, epsilon=0.4541
dp_eps_2, seed 52: epoch 05/30, loss=0.120649, epsilon=0.8392
dp_eps_2, seed 52: epoch 10/30, loss=0.101337, epsilon=1.1527
dp_eps_2, seed 52: epoch 15/30, loss=0.097682, epsilon=1.4043
dp_eps_2, seed 52: epoch 20/30, loss=0.091979, epsilon=1.6225
dp_eps_2, seed 52: epoch 25/30, loss=0.085943, epsilon=1.8187
dp_eps_2, seed 52: epoch 30/30, loss=0.081856, epsilon=1.9990
Ignoring stale cache: non_private, seed 62
non_private, seed 62: epoch 01/30, loss=0.158866
non_private, seed 62: epoch 05/30, loss=0.035032
non_private, seed 62: epoch 10/30, loss=0.022571
non_private, seed 62: epoch 15/30, loss=0.017309
non_private, seed 62: epoch 20/30, loss=0.015408
non_private, seed 62: epoch 25/30, loss=0.013689
non_private, seed 62: epoch 30/30, loss=0.012856
Ignoring stale cache: dp_eps_4, seed 62


...

dp_eps_4, seed 82: epoch 01/30, loss=0.259576, epsilon=1.3784
dp_eps_4, seed 82: epoch 05/30, loss=0.119032, epsilon=2.0538
dp_eps_4, seed 82: epoch 10/30, loss=0.103828, epsilon=2.5569
dp_eps_4, seed 82: epoch 15/30, loss=0.094702, epsilon=2.9719
dp_eps_4, seed 82: epoch 20/30, loss=0.088855, epsilon=3.3417
dp_eps_4, seed 82: epoch 25/30, loss=0.083605, epsilon=3.6813
dp_eps_4, seed 82: epoch 30/30, loss=0.084521, epsilon=3.9983
Ignoring stale cache: dp_eps_2, seed 82


/usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


dp_eps_2, seed 82: epoch 01/30, loss=0.274957, epsilon=0.4541
dp_eps_2, seed 82: epoch 05/30, loss=0.121145, epsilon=0.8392
dp_eps_2, seed 82: epoch 10/30, loss=0.106705, epsilon=1.1527
dp_eps_2, seed 82: epoch 15/30, loss=0.098383, epsilon=1.4043
dp_eps_2, seed 82: epoch 20/30, loss=0.093529, epsilon=1.6225
dp_eps_2, seed 82: epoch 25/30, loss=0.088147, epsilon=1.8187
dp_eps_2, seed 82: epoch 30/30, loss=0.089167, epsilon=1.9990


/tmp/ipykernel_477/2440085210.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  repeated_run_ids_results = pd.concat(


,condition,seed,actual_epsilon,threshold,recall,fnr,f1,fpr,pr_auc
2,non_private,42,NaN,0.29,0.708018,0.291982,0.814596,0.040058,0.935714
5,dp_eps_4,42,3.998267,0.02,0.726720,0.273280,0.810569,0.087736,0.897761
8,dp_eps_2,42,1.999038,0.03,0.685031,0.314969,0.784176,0.082072,0.895967
11,non_private,52,NaN,0.14,0.715032,0.284968,0.805124,0.080836,0.942404
14,dp_eps_4,52,3.998267,0.02,0.712538,0.287462,0.801508,0.086500,0.888878
17,dp_eps_2,52,1.999038,0.02,0.710356,0.289644,0.800281,0.085779,0.892525
20,non_private,62,NaN,0.24,0.740435,0.259565,0.835414,0.042529,0.946621
23,dp_eps_4,62,3.998267,0.02,0.723759,0.276241,0.807336,0.091443,0.894517
26,dp_eps_2,62,1.999038,0.03,0.714096,0.285904,0.803120,0.084852,0.893591
29,non_private,72,NaN,0.23,0.704824,0.295176,0.798182,0.080939,0.925761


## 10. Evaluate frozen primary and secondary attackers


In [35]:
# Primary and secondary policies are evaluated separately on identical target records.
repeated_run_mia_results = evaluate_policy(target_runs, fixed_attacks, "primary")
secondary_mia_results = evaluate_policy(target_runs, secondary_attacks, "secondary")
accounting_columns = [
    "condition", "seed", "run_source", "formal_dp", "target_epsilon",
    "actual_epsilon", "delta", "noise_multiplier",
]
repeated_run_mia_results = repeated_run_mia_results.merge(
    repeated_run_configs[accounting_columns], on=["condition", "seed"],
    how="left", validate="many_to_one",
)
secondary_mia_results = secondary_mia_results.merge(
    repeated_run_configs[accounting_columns], on=["condition", "seed"],
    how="left", validate="many_to_one",
)
repeated_run_mia_results.to_csv(RESULTS_DIR / "repeated_run_mia_results.csv", index=False)
secondary_mia_results.to_csv(RESULTS_DIR / "secondary_mia_results.csv", index=False)
display(repeated_run_mia_results[[
    "condition", "seed", "threat_model", "attack_model", *MIA_SUMMARY_METRICS,
]])


,condition,seed,threat_model,attack_model,mia_auc,mia_advantage,mia_balanced_accuracy,tpr_at_1pct_fpr,tpr_at_5pct_fpr
0,non_private,42,score_only_black_box,logistic_regression,0.501777,0.008970,0.502024,0.000000,0.000000
1,non_private,42,label_aware_audit,loss_threshold,0.501439,0.005636,0.500040,0.000000,0.000000
2,dp_eps_4,42,score_only_black_box,logistic_regression,0.502884,0.012860,0.500516,0.000000,0.000000
3,dp_eps_4,42,label_aware_audit,loss_threshold,0.501666,0.012225,0.504366,0.000000,0.000000
4,dp_eps_2,42,score_only_black_box,logistic_regression,0.502916,0.011749,0.500873,0.000000,0.000000
5,dp_eps_2,42,label_aware_audit,loss_threshold,0.501608,0.011193,0.503255,0.000000,0.000000
6,non_private,52,score_only_black_box,logistic_regression,0.504782,0.013495,0.501032,0.000000,0.000000
7,non_private,52,label_aware_audit,loss_threshold,0.502346,0.011828,0.501151,0.000000,0.000000
8,dp_eps_4,52,score_only_black_box,logistic_regression,0.503101,0.011114,0.501072,0.000000,0.000000
9,dp_eps_4,52,label_aware_audit,loss_threshold,0.501961,0.011114,0.500000,0.000000,0.000000


## 11. Across-seed uncertainty and matched-seed comparisons


In [36]:
def summarize_five(values: pd.Series) -> dict:
    numeric = pd.to_numeric(values, errors="raise").to_numpy(dtype=float)
    assert len(numeric) == len(RUN_SEEDS)
    assert np.isfinite(numeric).all()
    mean = float(np.mean(numeric))
    standard_deviation = float(np.std(numeric, ddof=1))
    half_width = T_CRITICAL_95_DF4 * standard_deviation / math.sqrt(len(numeric))
    return {
        "n_seeds": len(numeric),
        "mean": mean,
        "standard_deviation": standard_deviation,
        "ci_method": "two-sided t interval, df=4",
        "ci_low": mean - half_width,
        "ci_high": mean + half_width,
    }

tuned_ids = repeated_run_ids_results[
    (repeated_run_ids_results["split"] == "KDDTest+")
    & (
        repeated_run_ids_results["threshold_policy"]
        == "validation_selected_F2"
    )
].copy()

summary_rows = []
for condition in [spec["condition"] for spec in CONDITIONS]:
    condition_ids = tuned_ids[tuned_ids["condition"] == condition]
    for metric in IDS_SUMMARY_METRICS:
        summary_rows.append({
            "condition": condition,
            "domain": "IDS",
            "threat_model": None,
            "metric": metric,
            **summarize_five(condition_ids[metric]),
        })

    condition_mia = repeated_run_mia_results[
        repeated_run_mia_results["condition"] == condition
    ]
    for threat_model in ["score_only_black_box", "label_aware_audit"]:
        threat_rows = condition_mia[
            condition_mia["threat_model"] == threat_model
        ]
        for metric in MIA_SUMMARY_METRICS:
            summary_rows.append({
                "condition": condition,
                "domain": "MIA",
                "threat_model": threat_model,
                "metric": metric,
                **summarize_five(threat_rows[metric]),
            })

    condition_configs = repeated_run_configs[
        repeated_run_configs["condition"] == condition
    ]
    if condition != "non_private":
        summary_rows.append({
            "condition": condition,
            "domain": "privacy_accounting",
            "threat_model": None,
            "metric": "actual_epsilon",
            **summarize_five(condition_configs["actual_epsilon"]),
        })

repeated_run_summary = pd.DataFrame(summary_rows)
repeated_run_summary.to_csv(
    RESULTS_DIR / "repeated_run_summary.csv",
    index=False,
)
display(repeated_run_summary)


,condition,domain,threat_model,metric,n_seeds,mean,standard_deviation,ci_method,ci_low,ci_high
0,non_private,IDS,None,recall,5,0.709343,2.223883e-02,"two-sided t interval, df=4",0.681730,0.736956
1,non_private,IDS,None,fnr,5,0.290657,2.223883e-02,"two-sided t interval, df=4",0.263044,0.318270
2,non_private,IDS,None,f1,5,0.809750,1.614275e-02,"two-sided t interval, df=4",0.789706,0.829793
3,non_private,IDS,None,fpr,5,0.056101,2.274096e-02,"two-sided t interval, df=4",0.027865,0.084338
4,non_private,IDS,None,precision,5,0.943955,2.133293e-02,"two-sided t interval, df=4",0.917467,0.970444
5,non_private,IDS,None,pr_auc,5,0.937504,7.881250e-03,"two-sided t interval, df=4",0.927718,0.947290
6,non_private,IDS,None,threshold,5,0.206000,6.877500e-02,"two-sided t interval, df=4",0.120605,0.291395
7,non_private,MIA,score_only_black_box,mia_auc,5,0.502297,1.600786e-03,"two-sided t interval, df=4",0.500309,0.504284
8,non_private,MIA,score_only_black_box,mia_advantage,5,0.009415,2.482955e-03,"two-sided t interval, df=4",0.006332,0.012498
9,non_private,MIA,score_only_black_box,mia_balanced_accuracy,5,0.500397,1.107826e-03,"two-sided t interval, df=4",0.499021,0.501772


In [37]:
paired_rows = []
for comparison_condition in ["dp_eps_4", "dp_eps_2"]:
    for seed in RUN_SEEDS:
        reference_ids = tuned_ids[
            (tuned_ids["condition"] == "non_private")
            & (tuned_ids["seed"] == seed)
        ].iloc[0]
        comparison_ids = tuned_ids[
            (tuned_ids["condition"] == comparison_condition)
            & (tuned_ids["seed"] == seed)
        ].iloc[0]
        for metric in IDS_SUMMARY_METRICS:
            paired_rows.append({
                "reference_condition": "non_private",
                "comparison_condition": comparison_condition,
                "seed": seed,
                "domain": "IDS",
                "threat_model": None,
                "metric": metric,
                "reference_value": reference_ids[metric],
                "comparison_value": comparison_ids[metric],
                "difference_dp_minus_non_private": (
                    comparison_ids[metric] - reference_ids[metric]
                ),
            })

        for threat_model in ["score_only_black_box", "label_aware_audit"]:
            reference_mia = repeated_run_mia_results[
                (repeated_run_mia_results["condition"] == "non_private")
                & (repeated_run_mia_results["seed"] == seed)
                & (repeated_run_mia_results["threat_model"] == threat_model)
            ].iloc[0]
            comparison_mia = repeated_run_mia_results[
                (repeated_run_mia_results["condition"] == comparison_condition)
                & (repeated_run_mia_results["seed"] == seed)
                & (repeated_run_mia_results["threat_model"] == threat_model)
            ].iloc[0]
            for metric in MIA_SUMMARY_METRICS:
                paired_rows.append({
                    "reference_condition": "non_private",
                    "comparison_condition": comparison_condition,
                    "seed": seed,
                    "domain": "MIA",
                    "threat_model": threat_model,
                    "metric": metric,
                    "reference_value": reference_mia[metric],
                    "comparison_value": comparison_mia[metric],
                    "difference_dp_minus_non_private": (
                        comparison_mia[metric] - reference_mia[metric]
                    ),
                })

repeated_run_paired_differences = pd.DataFrame(paired_rows)
paired_summary_rows = []
paired_group_columns = [
    "reference_condition", "comparison_condition",
    "domain", "threat_model", "metric",
]
for keys, group in repeated_run_paired_differences.groupby(
    paired_group_columns,
    dropna=False,
):
    summary = summarize_five(group["difference_dp_minus_non_private"])
    paired_summary_rows.append({
        **dict(zip(paired_group_columns, keys)),
        **summary,
    })

repeated_run_paired_summary = pd.DataFrame(paired_summary_rows)
repeated_run_paired_differences.to_csv(
    RESULTS_DIR / "repeated_run_paired_differences.csv",
    index=False,
)
repeated_run_paired_summary.to_csv(
    RESULTS_DIR / "repeated_run_paired_summary.csv",
    index=False,
)
display(repeated_run_paired_summary)


,reference_condition,comparison_condition,domain,threat_model,metric,n_seeds,mean,standard_deviation,ci_method,ci_low,ci_high
0,non_private,dp_eps_2,IDS,NaN,f1,5,-0.015144,0.014891,"two-sided t interval, df=4",-0.033635,0.003346
1,non_private,dp_eps_2,IDS,NaN,fnr,5,0.008260,0.018598,"two-sided t interval, df=4",-0.014833,0.031353
2,non_private,dp_eps_2,IDS,NaN,fpr,5,0.027721,0.022023,"two-sided t interval, df=4",0.000376,0.055066
3,non_private,dp_eps_2,IDS,NaN,pr_auc,5,-0.046929,0.004982,"two-sided t interval, df=4",-0.053115,-0.040744
4,non_private,dp_eps_2,IDS,NaN,precision,5,-0.026924,0.020960,"two-sided t interval, df=4",-0.052948,-0.000899
5,non_private,dp_eps_2,IDS,NaN,recall,5,-0.008260,0.018598,"two-sided t interval, df=4",-0.031353,0.014833
6,non_private,dp_eps_2,IDS,NaN,threshold,5,-0.178000,0.066483,"two-sided t interval, df=4",-0.260550,-0.095450
7,non_private,dp_eps_2,MIA,label_aware_audit,mia_advantage,5,0.003445,0.002225,"two-sided t interval, df=4",0.000682,0.006208
8,non_private,dp_eps_2,MIA,label_aware_audit,mia_auc,5,0.000581,0.001481,"two-sided t interval, df=4",-0.001259,0.002420
9,non_private,dp_eps_2,MIA,label_aware_audit,mia_balanced_accuracy,5,-0.000135,0.002593,"two-sided t interval, df=4",-0.003354,0.003084


In [38]:
# Supplementary policy: same records/seeds, condition-specific shadow-selected attackers.
secondary_summary_rows = []
for (condition, threat_model), group in secondary_mia_results.groupby(["condition", "threat_model"]):
    for metric in MIA_SUMMARY_METRICS:
        secondary_summary_rows.append({
            "condition": condition, "threat_model": threat_model, "metric": metric,
            "analysis_role": "secondary", **summarize_five(group[metric]),
        })
secondary_mia_summary = pd.DataFrame(secondary_summary_rows)
secondary_pairs = []
for condition in ["dp_eps_4", "dp_eps_2"]:
    for threat_model in PRIMARY_ATTACK_MODELS:
        reference = secondary_mia_results[
            (secondary_mia_results["condition"] == "non_private")
            & (secondary_mia_results["threat_model"] == threat_model)
        ].set_index("seed")
        comparison = secondary_mia_results[
            (secondary_mia_results["condition"] == condition)
            & (secondary_mia_results["threat_model"] == threat_model)
        ].set_index("seed")
        for seed in RUN_SEEDS:
            for metric in MIA_SUMMARY_METRICS:
                secondary_pairs.append({
                    "comparison_condition": condition, "reference_condition": "non_private",
                    "threat_model": threat_model, "seed": seed, "metric": metric,
                    "difference_dp_minus_non_private": comparison.loc[seed, metric] - reference.loc[seed, metric],
                })
secondary_mia_paired_differences = pd.DataFrame(secondary_pairs)
secondary_pair_summaries = []
for (condition, threat_model, metric), group in secondary_mia_paired_differences.groupby(
    ["comparison_condition", "threat_model", "metric"]
):
    secondary_pair_summaries.append({
        "comparison_condition": condition, "reference_condition": "non_private",
        "threat_model": threat_model, "metric": metric, "analysis_role": "secondary",
        **summarize_five(group["difference_dp_minus_non_private"]),
    })
secondary_mia_paired_summary = pd.DataFrame(secondary_pair_summaries)
for name, table in {
    "secondary_mia_summary": secondary_mia_summary,
    "secondary_mia_paired_differences": secondary_mia_paired_differences,
    "secondary_mia_paired_summary": secondary_mia_paired_summary,
}.items():
    table.to_csv(RESULTS_DIR / f"{name}.csv", index=False)


## 12. Save evidence and run the final technical gate


In [39]:
all_warning_rows = []
for result in target_runs.values():
    all_warning_rows.extend(result.warning_rows)

warning_table = pd.DataFrame(
    all_warning_rows,
    columns=["stage", "condition", "seed", "category", "message", "occurrences"],
)
if not warning_table.empty:
    warning_table = (
        warning_table.groupby(
            ["stage", "condition", "seed", "category", "message"],
            dropna=False,
            as_index=False,
        )["occurrences"]
        .sum()
    )
warning_table.to_csv(RESULTS_DIR / "opacus_warning_summary.csv", index=False)

experiment_config = {
    "protocol_version": PROTOCOL_VERSION,
    "protocol_revision": PROTOCOL_REVISION,
    "experiment": "06_repeated_runs_stability",
    "seeds": RUN_SEEDS,
    "conditions": CONDITIONS,
    "seed_42_policy": (
        "Import accepted Experiment 05 target evidence; reconstruct scores from its "
        "verified model if needed. Stop if original evidence cannot be recovered."
    ),
    "architecture": {
        "target_input_dim": TARGET_INPUT_DIM,
        "hidden_dims": list(HIDDEN_DIMS),
        "activation": "ReLU",
        "output": "one logit",
        "batch_norm": False,
    },
    "training": {
        "optimizer": "Adam",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "loss": "BCEWithLogitsLoss",
    },
    "dp": {
        "target_delta": TARGET_DELTA,
        "max_grad_norm": MAX_GRAD_NORM,
        "accountant": ACCOUNTANT,
        "poisson_sampling": POISSON_SAMPLING,
        "secure_mode": SECURE_MODE,
        "privacy_scope": "DP-SGD optimisation conditional on fixed preprocessing.",
    },
    "mia": {
        "target_records": "Exact accepted Experiment 05 target MIA sample.",
        "attacker_source": "Accepted Experiment 05 condition-matched shadow data.",
        "primary_attacks": PRIMARY_ATTACK_MODELS,
        "secondary_selection_rule": "Highest Experiment 05 shadow-calibration AUC per condition and threat model; deterministic original-order tie-break.",
        "shadow_data_sha256": shadow_data_hashes,
        "seed42_reproduction_check": {
            "policy": "auc_absolute_1e-8_advantage_absolute_1e-10_v1",
            "auc_atol": 1e-8,
            "advantage_atol": 1e-10,
            "relative_tolerance": 0,
            "reason": "Observed 9.45e-9 LR AUC discrepancy; numerical origin plausible but not proven. Original estimates and differences preserved.",
        },
        "target_score_tuning": False,
        "new_shadow_models_trained": 0,
        "threat_models": ["score_only_black_box", "label_aware_audit"],
    },
    "uncertainty": {
        "unit": "target-training seed conditional on fixed split and frozen attackers",
        "ids_summary_metrics": IDS_SUMMARY_METRICS,
        "mia_summary_metrics": MIA_SUMMARY_METRICS,
        "pr_auc_definition": "average_precision_score, not trapezoidal PR area",
        "n_seeds": len(RUN_SEEDS),
        "interval": "two-sided t interval with df=4",
        "t_critical_95": T_CRITICAL_95_DF4,
        "paired_comparison": "DP minus non-private at each matched seed",
    },
    "execution": {
        "runtime": RUNTIME_IDENTITY,
        "resume": RESUME,
        "reuse_accepted_seed_42": REUSE_ACCEPTED_SEED_42,
    },
}
write_strict_json(RESULTS_DIR / "config.json", experiment_config)


In [40]:
expected_pairs = {
    (spec["condition"], seed)
    for spec in CONDITIONS
    for seed in RUN_SEEDS
}
completed_pairs = set(zip(
    repeated_run_configs["condition"],
    repeated_run_configs["seed"].astype(int),
))
run_gate = bool(
    len(repeated_run_configs) == 15
    and completed_pairs == expected_pairs
    and repeated_run_configs["cache_fingerprint"].notna().all()
)

dp_configs = repeated_run_configs[repeated_run_configs["formal_dp"] == True]
epsilon_gate = bool(
    len(dp_configs) == 10
    and dp_configs["actual_epsilon"].notna().all()
    and (
        (dp_configs["actual_epsilon"] - dp_configs["target_epsilon"]).abs()
        <= 0.10
    ).all()
    and np.allclose(
        dp_configs["delta"].astype(float),
        TARGET_DELTA,
        rtol=0,
        atol=1e-15,
    )
)

tuned_ids_gate = bool(
    len(tuned_ids) == 15
    and set(zip(tuned_ids["condition"], tuned_ids["seed"].astype(int)))
    == expected_pairs
    and tuned_ids[IDS_SUMMARY_METRICS].notna().all().all()
)

expected_mia_keys = {
    (spec["condition"], seed, threat_model)
    for spec in CONDITIONS
    for seed in RUN_SEEDS
    for threat_model in ["score_only_black_box", "label_aware_audit"]
}
observed_mia_keys = set(zip(
    repeated_run_mia_results["condition"],
    repeated_run_mia_results["seed"].astype(int),
    repeated_run_mia_results["threat_model"],
))
mia_gate = bool(
    len(repeated_run_mia_results) == 30
    and observed_mia_keys == expected_mia_keys
    and repeated_run_mia_results[
        MIA_SUMMARY_METRICS
    ].notna().all().all()
)

summary_gate = bool(
    len(repeated_run_summary) == len(CONDITIONS) * (len(IDS_SUMMARY_METRICS) + len(PRIMARY_ATTACK_MODELS) * len(MIA_SUMMARY_METRICS)) + 2
    and (repeated_run_summary["n_seeds"] == 5).all()
    and repeated_run_summary[
        ["mean", "standard_deviation", "ci_low", "ci_high"]
    ].notna().all().all()
)
paired_gate = bool(
    len(repeated_run_paired_differences) == 2 * len(RUN_SEEDS) * (len(IDS_SUMMARY_METRICS) + len(PRIMARY_ATTACK_MODELS) * len(MIA_SUMMARY_METRICS))
    and len(repeated_run_paired_summary) == 2 * (len(IDS_SUMMARY_METRICS) + len(PRIMARY_ATTACK_MODELS) * len(MIA_SUMMARY_METRICS))
    and (repeated_run_paired_summary["n_seeds"] == 5).all()
)
fixed_attacker_gate = bool(
    len(fixed_attacker_verification) == 12
    and fixed_attacker_verification["verification_passed"].all()
)
seed42_gate = bool(
    len(seed42_import_verification) == 24
    and seed42_import_verification["verification_passed"].all()
)

secondary_gate = bool(
    len(secondary_mia_results) == len(expected_mia_keys)
    and set(zip(secondary_mia_results["condition"], secondary_mia_results["seed"], secondary_mia_results["threat_model"])) == expected_mia_keys
    and np.isfinite(secondary_mia_results[MIA_SUMMARY_METRICS].to_numpy(dtype=float)).all()
    and len(secondary_mia_summary) == len(CONDITIONS) * 2 * len(MIA_SUMMARY_METRICS)
    and len(secondary_mia_paired_differences) == 2 * len(RUN_SEEDS) * 2 * len(MIA_SUMMARY_METRICS)
    and len(secondary_mia_paired_summary) == 2 * 2 * len(MIA_SUMMARY_METRICS)
)
finite_gate = bool(
    np.isfinite(tuned_ids[IDS_SUMMARY_METRICS].to_numpy(dtype=float)).all()
    and np.isfinite(repeated_run_mia_results[MIA_SUMMARY_METRICS].to_numpy(dtype=float)).all()
    and np.allclose(tuned_ids["recall"] + tuned_ids["fnr"], 1.0)
)
pre_manifest_outputs = [
    RESULTS_DIR / "secondary_mia_results.csv",
    RESULTS_DIR / "secondary_mia_summary.csv",
    RESULTS_DIR / "secondary_mia_paired_differences.csv",
    RESULTS_DIR / "secondary_mia_paired_summary.csv",
    RESULTS_DIR / "repeated_run_ids_results.csv",
    RESULTS_DIR / "repeated_run_mia_results.csv",
    RESULTS_DIR / "repeated_run_configs.csv",
    RESULTS_DIR / "repeated_run_summary.csv",
    RESULTS_DIR / "repeated_run_paired_differences.csv",
    RESULTS_DIR / "repeated_run_paired_summary.csv",
    RESULTS_DIR / "fixed_attacker_verification.csv",
    RESULTS_DIR / "seed42_import_verification.csv",
    RESULTS_DIR / "target_mia_sample_manifest.csv",
    RESULTS_DIR / "opacus_warning_summary.csv",
    RESULTS_DIR / "config.json",
]
outputs_gate = all(path.exists() for path in pre_manifest_outputs)

protocol_gates = {
    "preflight_gate": PREFLIGHT_PASSED,
    "secondary_gate": secondary_gate,
    "finite_gate": finite_gate,
    "run_gate": run_gate,
    "epsilon_gate": epsilon_gate,
    "tuned_ids_gate": tuned_ids_gate,
    "mia_gate": mia_gate,
    "summary_gate": summary_gate,
    "paired_gate": paired_gate,
    "fixed_attacker_gate": fixed_attacker_gate,
    "seed42_gate": seed42_gate,
    "outputs_gate": outputs_gate,
}
final_gate = all(protocol_gates.values())

print({
    **protocol_gates,
    "completed_condition_seed_runs": len(completed_pairs),
    "experiment_06_protocol_gate": final_gate,
})

if not final_gate:
    raise RuntimeError(
        "Experiment 06 protocol gate failed. Do not interpret or select a final DP setting."
    )

manifest = {
    **experiment_config,
    "created_at_unix": time.time(),
    "device": str(DEVICE),
    "dataset": {
        "train_file": str(TRAIN_FILE),
        "test_file": str(TEST_FILE),
        "train_sha256": actual_train_hash,
        "test_sha256": actual_test_hash,
    },
    "split": {
        "target_train_rows": len(target_train),
        "target_validation_rows": len(target_validation),
        "shadow_pool_rows": len(shadow_pool_indices),
        **split_hashes,
    },
    "preprocessor": {
        "path": str(PREPROCESSOR_FILE),
        "sha256": preprocessor_hash,
    },
    "prerequisites": {
        "experiment_05_manifest": str(EXP05_MANIFEST_FILE),
        "experiment_05_manifest_sha256": calculate_sha256(EXP05_MANIFEST_FILE),
        "experiment_05_config_sha256": calculate_sha256(EXP05_CONFIG_FILE),
        "experiment_05_attack_calibration_sha256": calculate_sha256(
            EXP05_ATTACK_CALIBRATION_FILE
        ),
        "target_mia_sample_sha256": target_sample_hash,
    },
    "target_run_configs": repeated_run_configs.to_dict(orient="records"),
    "fixed_attackers": fixed_attacker_verification.to_dict(orient="records"),
    "protocol_gates": protocol_gates,
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "torch": torch.__version__,
        "opacus": __import__("opacus").__version__,
    },
    "required_outputs": [str(path) for path in pre_manifest_outputs],
    "output_sha256": {path.name: calculate_sha256(path) for path in pre_manifest_outputs},
}

manifest_path = RESULTS_DIR / "repeated_runs_manifest.json"
write_strict_json(manifest_path, manifest)
write_strict_json(MANIFEST_DIR / "repeated_runs_manifest.json", manifest)

evidence_files = [*pre_manifest_outputs, manifest_path]
assert all(path.exists() for path in evidence_files)

evidence_zip = RESULTS_DIR / "experiment06_evidence.zip"
with zipfile.ZipFile(evidence_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in evidence_files:
        archive.write(path, arcname=path.name)

print("Experiment 06 protocol gate: PASSED")
print("Evidence ZIP:", evidence_zip)

try:
    from google.colab import files
    files.download(str(evidence_zip))
except Exception:
    print("Download manually from:", evidence_zip)


{'preflight_gate': True, 'secondary_gate': True, 'finite_gate': True, 'run_gate': True, 'epsilon_gate': True, 'tuned_ids_gate': True, 'mia_gate': True, 'summary_gate': True, 'paired_gate': True, 'fixed_attacker_gate': True, 'seed42_gate': True, 'outputs_gate': True, 'completed_condition_seed_runs': 15, 'experiment_06_protocol_gate': True}
Experiment 06 protocol gate: PASSED
Evidence ZIP: /content/drive/MyDrive/ML-DP-NID/results/repeated_runs/experiment06_evidence.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Interpretation boundary

- Treat target-training seed as the uncertainty unit; the notebook reports mean, sample standard deviation, and a two-sided 95% t interval across five seeds.
- Use matched-seed DP-minus-non-private differences when discussing changes.
- ε≈4 becomes the final balance candidate only if its utility pattern is stable across seeds.
- Near-chance MIA results do not prove that a model is private.
- Do not claim a measured leakage reduction unless the relevant repeated comparison supports it.
- Do not add ε≈1, another dataset, federated learning, transformers, or a new privacy mechanism here.

- Report false-positive rate, precision, and selected threshold alongside Recall/FNR.
- `pr_auc` is average precision. Primary and secondary attacker results are separate.
- Five-seed t intervals describe training variability, not independent dataset replication.
- Seed 42 originates in Experiment 05; its original software/device provenance is retained.
- The ε≈4 candidate was identified after inspecting the sweep; do not present this as fresh
  held-out confirmation of configuration selection or a universal operational optimum.
- Send the evidence ZIP for review before final frontier analysis or optional external validation.
